# EDA — Demanda horaria de bicicletas 🚲

Exploración del dataset **procesado** (`data/processed/bikeshare_features.parquet`), que integra
las 3 fuentes: **UCI Bike Sharing** + **clima real (Open-Meteo)** + **feriados (holidays)**.


In [1]:
import pandas as pd
import plotly.express as px
from bikeshare.etl.load import load_processed

df = load_processed()
print(df.shape)
df[["datetime", "cnt", "temperature_2m", "precipitation", "is_holiday", "hour"]].head()

(17158, 42)


,datetime,cnt,temperature_2m,precipitation,is_holiday,hour
0,2011-01-02 00:00:00,17,13.1,0.2,0,0
1,2011-01-02 01:00:00,17,13.1,0.2,0,1
2,2011-01-02 02:00:00,9,13.2,0.0,0,2
3,2011-01-02 03:00:00,6,13.6,0.0,0,3
4,2011-01-02 04:00:00,3,13.6,0.1,0,4


## 1. Estadísticos de la demanda (`cnt`)

In [2]:
df['cnt'].describe().to_frame().T

,count,mean,std,min,25%,50%,75%,max
cnt,17158.0,191.451976,181.504947,1.0,42.0,145.0,283.0,977.0


## 2. Demanda promedio por hora del día

In [3]:
by_hour = df.groupby('hour', as_index=False)['cnt'].mean()
fig = px.line(by_hour, x='hour', y='cnt', markers=True,
              title='Demanda promedio por hora del día',
              labels={'cnt': 'Demanda media', 'hour': 'Hora'})
fig.show()

## 3. Heatmap hora × día de la semana

In [4]:
dias = ['Lun','Mar','Mié','Jue','Vie','Sáb','Dom']
pivot = df.pivot_table(index='hour', columns='dayofweek', values='cnt', aggfunc='mean')
pivot.columns = [dias[c] for c in pivot.columns]
fig = px.imshow(pivot, aspect='auto', color_continuous_scale='Viridis',
                labels=dict(x='Día', y='Hora', color='Demanda media'),
                title='Demanda media por hora y día de la semana')
fig.show()

## 4. Relación clima ↔ demanda

In [5]:
fig = px.scatter(df.sample(3000, random_state=42), x='temperature_2m', y='cnt',
                 color='precipitation', color_continuous_scale='Blues_r',
                 opacity=0.5, title='Temperatura vs demanda (color = precipitación)',
                 labels={'temperature_2m': 'Temperatura (°C)', 'cnt': 'Demanda'})
fig.show()

## 5. Demanda: días laborales vs feriados

In [6]:
g = df.groupby('is_holiday')['cnt'].mean().rename({0: 'Normal', 1: 'Feriado'})
fig = px.bar(g, title='Demanda media: día normal vs feriado',
             labels={'value': 'Demanda media', 'is_holiday': ''})
fig.update_layout(showlegend=False)
fig.show()

## Conclusiones del EDA
- La demanda tiene **doble peak** (mañana y tarde) en días laborales → patrón de *commuting*.
- **Fin de semana / feriados** muestran un patrón más plano y desplazado al mediodía.
- La **temperatura** se relaciona positivamente con la demanda; la **precipitación** la reduce.
- Estos patrones justifican las features **cíclicas**, de **calendario** y **autoregresivas** del modelo.
